# PEMS-SF 1D-CNN 训练笔记 (V2.2.0 - 7类全周 + 梯度特征)
目标： 利用 1D-CNN 的局部特征提取能力，结合原始流量与梯度信息，解决周二/周三混淆问题。

数据： PEMS-SF，双通道输入 (Raw Flow + Gradient)。

集成了之前讨论的所有增强手段：  

特征融合 (Feature Fusion): 输入双通道序列 (Flow + Gradient) + 统计特征 (Mean, Std, Max)。  

SE-Block: 增强 CNN 对关键特征的注意力。  

Focal Loss: 解决样本不平衡，专治“周三隐形”问题。  

数据增强 (Data Augmentation): 训练时加入随机噪声，提高泛化能力。  

强正则化: Dropout 提高到 0.5，防止过拟合。  

正确的数据集划分: 确保训练集有噪声，验证集/测试集干净。

In [ ]:
# ==========================================
# Cell 1: 基础依赖与配置
# ==========================================
import os, pathlib, json, math, random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, Subset
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# 配置
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
ROOT = pathlib.Path('.')

# 输出目录 V2.2.0
CM_DIR = ROOT / 'confusion_matrices220'
CM_DIR.mkdir(exist_ok=True)
MODEL_DIR = ROOT / 'models220' 
MODEL_DIR.mkdir(exist_ok=True)

# 随机种子 - 保证可复现性
def set_seed(seed=42):
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)
print(f'Device: {DEVICE}')
print(f'Output Dir: {CM_DIR}')

In [ ]:
# 训练与解释的可复现性与确定性设置
torch.manual_seed(0); np.random.seed(0); random.seed(0)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [ ]:
# ==========================================
# Cell 2: 数据读取、预处理与增强
# ==========================================

# 路径配置
train_path = ROOT / 'data' / 'PEMS_train'
test_path  = ROOT / 'data' / 'PEMS_test'
labels_train_path = ROOT / 'data' / 'PEMS_trainlabels'
labels_test_path  = ROOT / 'data' / 'PEMS_testlabels'
stations_list_path = ROOT / 'data' / 'stations_list'
groups_index_path = ROOT / 'data' / 'processed' / 'pems_sf_groups_index.json'

# 基础解析函数
def parse_day_matrix(line: str):
    line = line.strip()
    if line.startswith('[') and line.endswith(']'): line = line[1:-1]
    rows = [r.strip() for r in line.split(';') if r.strip()]
    data = []
    for r in rows:
        nums = [float(x) for x in r.split() if x.replace('.', '', 1).isdigit()]
        if nums: data.append(nums)
    if not data: return None
    arr = np.full((len(data), max(len(rr) for rr in data)), np.nan, dtype=float)
    for i, rr in enumerate(data): arr[i, :len(rr)] = rr
    return arr

def iter_day_matrices(path: pathlib.Path, limit=None):
    with path.open('r', encoding='utf-8', errors='ignore') as f:
        for i, line in enumerate(f):
            if limit is not None and i >= limit: break
            mat = parse_day_matrix(line)
            yield mat if mat is not None else None

def load_labels(path: pathlib.Path):
    txt = path.read_text(encoding='utf-8', errors='ignore').strip().strip('[]')
    return np.array([int(x) for x in txt.split() if x.isdigit()], dtype=int)

# 加载元数据
labels_train = load_labels(labels_train_path)
labels_test  = load_labels(labels_test_path)
station_ids = [int(x) for x in stations_list_path.read_text(encoding='utf-8', errors='ignore').strip().strip('[]').split() if x.isdigit()]

# 加载分组掩码
group_station_masks = {}
if groups_index_path.exists():
    processed = json.loads(groups_index_path.read_text(encoding='utf-8'))
    groups_index = processed.get('groups_index', {})
    sid_to_pos = {sid: i for i, sid in enumerate(station_ids)}
    for g in ['g1','g2','g3','g4','g5']:
        mask = np.zeros(len(station_ids), dtype=bool)
        for sid in groups_index.get(g, []):
            if sid in sid_to_pos: mask[sid_to_pos[sid]] = True
        group_station_masks[g] = mask
    print('分组掩码加载完毕。')
else:
    print('警告：未找到分组文件，将使用全量数据。')
    group_station_masks = {'all': np.ones(len(station_ids), dtype=bool)}

# === 核心修改：曲线清洗与特征工程 (含统计特征) ===
def _process_curve(curve: np.ndarray):
    """
    输入: 原始单条曲线 (144,)
    输出: result (2, 144), stats (3,)
    """
    # 1. 长度对齐与插值
    curve = curve[:144]
    if curve.shape[0] < 144:
        curve = np.pad(curve, (0, 144 - curve.shape[0]), mode='constant', constant_values=np.nan)
    idx = np.arange(curve.size)
    mask = ~np.isnan(curve)
    if mask.any():
        curve = np.interp(idx, idx[mask], curve[mask])
    curve = np.nan_to_num(curve, nan=0.0)

    # 2. 归一化 (Min-Max 到 0-1)
    min_val, max_val = curve.min(), curve.max()
    denom = max_val - min_val + 1e-8
    norm_curve = (curve - min_val) / denom
    
    # === 计算统计特征 (Fusion) ===
    mean_val = np.mean(norm_curve)
    std_val  = np.std(norm_curve)
    # 使用 log1p 压缩最大值特征，保留量级信息
    log_max_val = np.log1p(max_val) / 10.0 
    stats = np.array([mean_val, std_val, log_max_val], dtype=np.float32)

    # 3. 计算梯度 (Gradient)
    grad_curve = np.gradient(norm_curve)
    
    # 4. 堆叠通道
    result = np.stack([norm_curve, grad_curve], axis=0)
    
    return result.astype(np.float32), stats

# Dataset 定义 (支持数据增强)
class PEMS_CNNDataset(Dataset):
    def __init__(self, split_path: pathlib.Path, labels: np.ndarray, use_mask: np.ndarray, augment=False):
        self.samples = []
        self.augment = augment # 是否开启增强
        
        for day_i, mat in enumerate(iter_day_matrices(split_path)):
            if day_i >= len(labels): break
            if mat is None or mat.shape[1] < 144: continue
            
            original_label = int(labels[day_i])
            # 7分类：原始标签 1-7 -> 映射为 0-6
            y = original_label - 1 
            if y < 0 or y > 6: continue 
            
            if use_mask.shape[0] == mat.shape[0]:
                mat = mat[use_mask, :]
            
            for sidx in range(mat.shape[0]):
                two_channel_seq, stats = _process_curve(mat[sidx, :144])
                if not np.isfinite(two_channel_seq).all(): continue
                self.samples.append((two_channel_seq, stats, y))
        
        self.n = len(self.samples)
    
    def __len__(self): return self.n
    
    def __getitem__(self, idx):
        seq, stats, y = self.samples[idx]
        seq_tensor = torch.from_numpy(seq)
        
        # === 数据增强 (仅在训练时触发) ===
        if self.augment:
            # 随机高斯噪声 (Noise Injection)
            # 强度 0.02 (约 2% 的波动)
            noise = torch.randn_like(seq_tensor) * 0.02
            seq_tensor = seq_tensor + noise
            
        return seq_tensor, torch.from_numpy(stats), torch.tensor(y, dtype=torch.long)

In [ ]:
# ==========================================
# Cell 3: 构建DataLoader (V2.2.0)
# ==========================================
group_loaders = {}
for g, mask in group_station_masks.items():
    print(f'准备分组 {g} 数据...')
    
    # 1. 实例化两个数据集：一个带增强(Train)，一个不带(Val)
    # 它们读取同样的文件，但 __getitem__ 行为不同
    ds_train_source = PEMS_CNNDataset(train_path, labels_train, mask, augment=True)
    ds_val_source   = PEMS_CNNDataset(train_path, labels_train, mask, augment=False)
    
    # 2. 生成随机划分的索引
    total_len = len(ds_train_source)
    n_train = int(total_len * 0.8)
    n_val = total_len - n_train
    
    # 生成随机排列的索引
    indices = torch.randperm(total_len, generator=torch.Generator().manual_seed(42)).tolist()
    train_indices = indices[:n_train]
    val_indices = indices[n_train:]
    
    # 3. 创建 Subset
    # 训练集用 ds_train_source (有噪声)，验证集用 ds_val_source (无噪声)
    train_subset = Subset(ds_train_source, train_indices)
    val_subset   = Subset(ds_val_source, val_indices)
    
    # Test (绝对不能增强)
    ds_test = PEMS_CNNDataset(test_path, labels_test, mask, augment=False)
    
    group_loaders[g] = {
        'train': DataLoader(train_subset, batch_size=64, shuffle=True, num_workers=0),
        'val':   DataLoader(val_subset, batch_size=128, shuffle=False, num_workers=0),
        'test':  DataLoader(ds_test, batch_size=128, shuffle=False, num_workers=0)
    }
    print(f'  - Train: {len(train_subset)} (Augmented), Val: {len(val_subset)} (Clean), Test: {len(ds_test)}')

## 模型定义：模型定义 (SE + Fusion + High Dropout)

In [ ]:
# ==========================================
# Cell 4: 1D-CNN + SE + Fusion (V2.2.0)
# ==========================================
class SEBlock(nn.Module):
    def __init__(self, channel, reduction=16):
        super(SEBlock, self).__init__()
        self.avg_pool = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Sequential(
            nn.Linear(channel, channel // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(channel // reduction, channel, bias=False),
            nn.Sigmoid()
        )
    def forward(self, x):
        b, c, _ = x.size()
        y = self.avg_pool(x).view(b, c)
        y = self.fc(y).view(b, c, 1)
        return x * y.expand_as(x)

class Simple1DCNN(nn.Module):
    def __init__(self, in_channels=2, num_classes=7):
        super(Simple1DCNN, self).__init__()
        
        # Block 1: 捕捉大尺度特征
        self.block1 = nn.Sequential(
            nn.Conv1d(in_channels, 32, 7, padding=3),
            nn.BatchNorm1d(32), nn.ReLU(), nn.MaxPool1d(2)
        )
        self.se1 = SEBlock(32, reduction=4) 

        # Block 2: 捕捉中尺度特征
        self.block2 = nn.Sequential(
            nn.Conv1d(32, 64, 5, padding=2),
            nn.BatchNorm1d(64), nn.ReLU(), nn.MaxPool1d(2)
        )
        self.se2 = SEBlock(64, reduction=8)

        # Block 3: 捕捉微观细节
        self.block3 = nn.Sequential(
            nn.Conv1d(64, 128, 3, padding=1),
            nn.BatchNorm1d(128), nn.ReLU()
        )
        self.se3 = SEBlock(128, reduction=16)

        self.gap = nn.AdaptiveAvgPool1d(1)
        
        # Fusion Head
        # CNN特征(128) + 统计特征(3) = 131
        self.fc = nn.Sequential(
            nn.Dropout(0.5), # <--- 提高 Dropout 到 0.5 (强正则化)
            nn.Linear(131, 64),
            nn.ReLU(),
            nn.Linear(64, num_classes)
        )
        
    def forward(self, x, stats):
        # x: (B, 2, 144)
        x = self.block1(x); x = self.se1(x)
        x = self.block2(x); x = self.se2(x)
        x = self.block3(x); x = self.se3(x)
        
        x = self.gap(x).view(x.size(0), -1) # (B, 128)
        
        # 特征融合
        combined = torch.cat([x, stats], dim=1) # (B, 131)
        
        logits = self.fc(combined)
        return logits

## 训练循环 (Focal Loss + Early Stopping)

In [ ]:
# ==========================================
# Cell 5: 训练循环 (V2.2.0 - Focal Loss)
# ==========================================
class FocalLoss(nn.Module):
    def __init__(self, alpha=1, gamma=2, reduction='mean'):
        super(FocalLoss, self).__init__()
        self.gamma = gamma
        self.reduction = reduction
        self.ce = nn.CrossEntropyLoss(reduction='none', label_smoothing=0.1)

    def forward(self, inputs, targets):
        logpt = -self.ce(inputs, targets)
        pt = torch.exp(logpt)
        # 核心公式：降低易分样本权重
        loss = (1 - pt) ** self.gamma * -logpt
        if self.reduction == 'mean': return loss.mean()
        return loss.sum()

def train_model(loaders, group_name, epochs=60):
    model = Simple1DCNN(in_channels=2, num_classes=7).to(DEVICE)
    optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-3) # 增加正则化
    
    # 使用 Focal Loss
    criterion = FocalLoss(gamma=2.0).to(DEVICE)
    
    best_acc = 0.0
    best_state = None
    
    # 早停机制
    patience = 12 # 稍微宽容一点，因为有噪声，Loss波动可能大
    counter = 0
    
    for epoch in range(1, epochs+1):
        model.train()
        total_loss = 0
        for x, stats, y in loaders['train']:
            x, stats, y = x.to(DEVICE), stats.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad()
            out = model(x, stats)
            loss = criterion(out, y)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
            
        # Validation (Clean Data)
        model.eval()
        correct = 0
        total = 0
        with torch.no_grad():
            for x, stats, y in loaders['val']:
                x, stats, y = x.to(DEVICE), stats.to(DEVICE), y.to(DEVICE)
                out = model(x, stats)
                preds = torch.argmax(out, dim=1)
                correct += (preds == y).sum().item()
                total += y.size(0)
        
        val_acc = correct / total if total > 0 else 0
        
        if val_acc > best_acc:
            best_acc = val_acc
            best_state = model.state_dict()
            counter = 0 # 重置早停
        else:
            counter += 1
            
        if epoch % 5 == 0:
            print(f"[{group_name}] Epoch {epoch}/{epochs} | Loss: {total_loss/len(loaders['train']):.4f} | Val Acc: {val_acc:.2%}")
        
        if counter >= patience:
            print(f"[{group_name}] Early stopping at epoch {epoch}")
            break
            
    return best_state, best_acc

# 开始训练
group_models = {}
for g, loaders in group_loaders.items():
    print(f"\n>>> 开始训练分组: {g}")
    # Epochs 设为 60，配合 Early Stopping
    state, acc = train_model(loaders, g, epochs=60) 
    group_models[g] = state
    print(f"[{g}] 训练完成，最佳验证准确率: {acc:.2%}")
    if state:
        torch.save(state, MODEL_DIR / f'cnn_v220_{g}.pth')

print("\n所有模型训练结束。")

## 评估与阈值
可视化 - 组合混淆矩阵 (V2.2.0)

In [ ]:
# ==========================================
# Cell 6: 可视化 - 组合混淆矩阵 (V2.2.0)
# ==========================================

def get_predictions(model_state, loader):
    model = Simple1DCNN(in_channels=2, num_classes=7).to(DEVICE)
    model.load_state_dict(model_state)
    model.eval()
    
    y_true = []
    y_pred = []
    
    with torch.no_grad():
        for x, stats, y in loader:
            x = x.to(DEVICE)
            stats = stats.to(DEVICE)
            logits = model(x, stats)
            preds = torch.argmax(logits, dim=1)
            y_true.extend(y.cpu().numpy())
            y_pred.extend(preds.cpu().numpy())
            
    return np.array(y_true), np.array(y_pred)

def plot_combined_matrix(mode='test'):
    print(f"正在生成 {mode} 集混淆矩阵组合图 (V2.2.0)...")
    
    layout = {
        'g1': (0, 0), 'g2': (0, 1), 'g3': (0, 2),
        'g4': (1, 0), 'g5': (1, 1), 'global': (1, 2)
    }
    
    fig, axes = plt.subplots(2, 3, figsize=(20, 14))
    labels = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
    
    global_true = []
    global_pred = []
    
    for g in ['g1', 'g2', 'g3', 'g4', 'g5']:
        if g not in group_models: continue
        
        loader = group_loaders[g][mode]
        y_t, y_p = get_predictions(group_models[g], loader)
        global_true.extend(y_t)
        global_pred.extend(y_p)
        
        cm = confusion_matrix(y_t, y_p)
        cm_norm = cm.astype('float') / (cm.sum(axis=1)[:, np.newaxis] + 1e-8)
        
        row, col = layout[g]
        ax = axes[row, col]
        sns.heatmap(cm_norm, annot=True, fmt='.2%', cmap='Blues',
                    xticklabels=labels, yticklabels=labels, cbar=False, ax=ax)
        ax.set_title(f"{g.upper()} ({mode.capitalize()})", fontsize=14, fontweight='bold')
        ax.set_ylabel('True')
        ax.set_xlabel('Predicted')

    # Global
    if len(global_true) > 0:
        cm_glob = confusion_matrix(global_true, global_pred)
        cm_glob_norm = cm_glob.astype('float') / (cm_glob.sum(axis=1)[:, np.newaxis] + 1e-8)
        
        row, col = layout['global']
        ax = axes[row, col]
        sns.heatmap(cm_glob_norm, annot=True, fmt='.2%', cmap='Blues',
                    xticklabels=labels, yticklabels=labels, cbar=True, ax=ax)
        ax.set_title(f"Global All Groups ({mode.capitalize()})", fontsize=14, fontweight='bold')
        ax.set_ylabel('True')
        ax.set_xlabel('Predicted')
    
    plt.suptitle(f'{mode.capitalize()} Set Confusion Matrices (Final Fusion V2.2.0)', 
                 fontsize=18, fontweight='bold', y=0.98)
    plt.tight_layout(rect=[0, 0, 1, 0.96])
    save_path = CM_DIR / f'combined_{mode}_matrices_v220.png'
    plt.savefig(save_path, dpi=300)
    print(f"图表已保存: {save_path}")
    plt.close()

# 执行绘图
plot_combined_matrix('val')
plot_combined_matrix('test')